# Keep `foundation_model_with_gateway` Warm

**Run this notebook at the start of the workshop and leave it running.**

The `foundation_model_with_gateway` endpoint scales to zero after a period of inactivity.
A cold start takes 3-5 minutes — long enough to kill the momentum during a live demo.
This notebook sends a lightweight ping every 30 seconds to keep the endpoint hot.

**To stop:** Click **Cancel** (▪) in the notebook toolbar, or interrupt the kernel.

> Tip: Run this in a separate browser tab so you can work in other notebooks without interrupting it.

In [ ]:
dbutils.widgets.text("endpoint_name", "foundation_model_with_gateway", "Endpoint Name")
dbutils.widgets.text("interval_seconds", "30", "Ping Interval (seconds)")

endpoint_name    = dbutils.widgets.get("endpoint_name")
interval_seconds = int(dbutils.widgets.get("interval_seconds"))

print(f"Endpoint : {endpoint_name}")
print(f"Interval : every {interval_seconds}s")

In [ ]:
import time
import requests
from datetime import datetime, timezone
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
workspace_url = w.config.host
endpoint_url  = f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations"

headers = {"Content-Type": "application/json"}
headers.update(w.config.authenticate())

ping_payload = {
    "messages": [{"role": "user", "content": "Hello"}],
    "max_tokens": 5,
}

print(f"Pinging {endpoint_url}")
print(f"Press Cancel (▪) in the toolbar to stop.\n")
print(f"{'Time (UTC)':<26} {'Status':>8} {'Latency':>10} {'Note'}")
print("-" * 68)

ping_count = 0
while True:
    ts      = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    t0      = time.time()
    note    = ""

    try:
        resp    = requests.post(endpoint_url, headers=headers, json=ping_payload, timeout=60)
        latency = time.time() - t0
        status  = resp.status_code

        if status == 200:
            note = "OK"
        elif status == 429:
            note = "rate-limited (harmless)"
        else:
            body = resp.json()
            note = body.get("message", body.get("error", ""))[:50]

    except requests.exceptions.Timeout:
        latency = time.time() - t0
        status  = "TIMEOUT"
        note    = "endpoint may be scaling up — will retry"
    except Exception as e:
        latency = time.time() - t0
        status  = "ERROR"
        note    = str(e)[:50]

    ping_count += 1
    print(f"{ts:<26} {str(status):>8} {latency:>9.1f}s  {note}")

    time.sleep(interval_seconds)